# HoCVid End-to-End Training

This notebook orchestrates the training loop for the completed `HocVidComplete` multi-modal image restoration pipeline. 

### Phases of Training Supported:
1. **Stage 1 (Adapters Only):** Train the fusion adapters and the M2Restore decoder head while keeping all massive backbones (AdaIR, ViT, SAM, MiDaS) strictly frozen.
2. **Stage 2 (Fine-tuning):** Unfreeze the DDER MoE router for task-specific routing optimization.
3. **Stage 3 (Full End-to-End):** Unfreeze the AFLB FreModule for absolute maximal performance.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# Import our fully assembled pipeline
from hocvid_complete import HocVidComplete

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

## 1. Dataset & DataLoader Setup
Define your custom dataset here. *Tip: If you run out of GPU Memory (VRAM), pre-compute the SAM and MiDaS maps and load them directly from disk in the dataset instead of running them live during training!*

In [ ]:
class HoCVidDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.data_dir = data_dir
        self.is_train = is_train
        # TODO: Load your image paths here
        self.image_paths = [] # Example: ['img1.png', 'img2.png']
        
    def __len__(self):
        return max(len(self.image_paths), 100) # Dummy length of 100 for testing the loop
    
    def __getitem__(self, idx):
        # TODO: Implement actual image loading and augmentation (e.g., using PIL or OpenCV)
        # Replace this dummy data with your real dataset logic.
        
        # 1. Load Degraded Image
        degraded_img = torch.rand(3, 256, 256) # [0, 1] range
        
        # 2. Load Target Clean Image
        clean_img = torch.rand(3, 256, 256)    # [0, 1] range
        
        # 3. Load Degradation Class (One-hot for DDER MoE, or None)
        de_cls = torch.zeros(6)
        de_cls[0] = 1.0 # E.g., class 0 is haze
        
        # 4. Optional: Load Pre-computed SAM/MiDaS maps to save massive VRAM
        sam_map = torch.rand(1, 256, 256)
        midas_map = torch.rand(1, 256, 256)
        
        return {
            'degraded': degraded_img,
            'clean': clean_img,
            'de_cls': de_cls,
            'sam': sam_map,
            'midas': midas_map
        }

# Hyperparameters
BATCH_SIZE = 2
NUM_WORKERS = 4

# Create Loaders
train_dataset = HoCVidDataset(data_dir="path/to/train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

print(f"Batches per epoch: {len(train_loader)}")

## 2. Model, Loss, and Optimizer Initialization

In [ ]:
# Initialize Model
model = HocVidComplete(
    dder_weights_path='DDER/dder.pt',
    use_sam=True,
    use_midas=True
).to(device)

# By default, the model enforces Stage 1 freeze policies (~10M parameters active)
model.freeze_upstream()

# If you want Stage 2/3, uncomment these:
# model.unfreeze_stage2() 
# model.unfreeze_stage3()

model.get_parameter_summary()

# Initialize Optimizer with differential Learning Rates from our wrapper
BASE_LR = 2e-4
optimizer = optim.AdamW(model.get_param_groups(base_lr=BASE_LR), weight_decay=1e-4)

# Initialize Mixed Precision Scaler for speed and memory
scaler = GradScaler()

# Base Loss Function (L1 is standard for Restoration)
criterion_l1 = nn.L1Loss()

## 3. The Training Loop

In [ ]:
EPOCHS = 50
SAVE_DIR = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        # 1. Load data to GPU
        degraded = batch['degraded'].to(device)
        clean = batch['clean'].to(device)
        de_cls = batch['de_cls'].to(device)
        sam = batch['sam'].to(device)
        midas = batch['midas'].to(device)
        
        optimizer.zero_grad()
        
        # 2. Forward Pass with AMP (Automatic Mixed Precision)
        # Speeds up ViT and Transformer components heavily
        with autocast():
            restored, losses_dict = model(
                degraded_input=degraded,
                de_cls=de_cls,
                sam_boundary=sam,
                midas_depth=midas,
                return_losses=True
            )
            
            # Compute Image Loss (L1) + Internal Routing Loss (MoE)
            loss_img = criterion_l1(restored, clean)
            loss_moe = losses_dict.get('moe_routing_loss', 0.0)
            
            total_loss = loss_img + (0.1 * loss_moe) # Weight MoE loss
            
        # 3. Backward Pass & Optimize via Scaler
        scaler.scale(total_loss).backward()
        
        # Unscale before clipping to prevent explosive gradients in adapters
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.get_trainable_params(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        # Logging
        epoch_loss += total_loss.item()
        pbar.set_postfix({'Loss': f"{total_loss.item():.4f}"})
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch} Completed | Average Loss: {avg_loss:.4f}")
    
    # Save Checkpoint
    if epoch % 5 == 0 or epoch == EPOCHS:
        ckpt_path = os.path.join(SAVE_DIR, f"hocvid_epoch_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }, ckpt_path)
        print(f"Saved checkpoint -> {ckpt_path}")